In [ ]:
# Import
import numpy as np
import pandas as pd
import re
import xarray as xr
import matplotlib.pyplot as plt
import datetime, os
from cartopy import crs as ccrs
from cartopy import feature as cfeature

#### Step 1:
- Make sure all LogBook IDs are properly named. We fix variations between logbook names in the logbooks summary csv and the "LogBook ID" column of our log entries field. These discrepancies are caused by: lengty logbook IDs which were truncated in the entries file, variations in capitalization, and date ranges which were entered differently between the logbook and log entries files.
- Calculate summary stats to give an overview of the data contained by each logbook and its origins.

In [ ]:
#read in both cleaned datasats - tier 1 and 4
base_dir = os.getcwd()

#Readin in cleaned datasets
export_finalized = {}
for i in range (4):
    print(f"Loaded: {str(os.path.join(base_dir, f'csv_files/Tier{i+1}.csv'))}")
    export_finalized[i+1] = pd.read_csv(os.path.join(base_dir, f'csv_files/Tier{i+1}.csv'), low_memory=False)
    print
    export_finalized[i+1]['Entry Date'] = pd.to_datetime(export_finalized[i+1]['Entry Date'])

In [ ]:
export_finalized[i+1].columns

In [ ]:
# #we will use tier 4
# tier = 4
# df = export_finalized[tier].copy()

In [ ]:
processed_tiers = {}

for i in range(4):
    tier = i + 1
    current_df = export_finalized[tier].copy()
    
    # create mask
    trunc_mask = current_df["LogBook ID"].astype(str).str.contains("…")
    trunc_ids = current_df.loc[trunc_mask, "LogBook ID"].unique()
    
    print(f"\n--- Tier {tier} ---")
    print(f"There are {len(trunc_ids)} truncated IDs found in the dataset.")
    
    from utils.meta import logbook_id_fix_map
    
    # check that we are correcting all truncations logbook IDs
    if len(trunc_ids) != len(logbook_id_fix_map):
        print("WARNING: Number of truncated IDs does NOT match number of entries in logbook_id_fix_map!")
        print(f"  - Truncated IDs found: {len(trunc_ids)}")
        print(f"  - Entries in fix map:   {len(logbook_id_fix_map)}")
    
        # extra diagnostics
        print("\nIDs in the dataset but NOT in your map:")
        missing_from_map = set(trunc_ids) - set(logbook_id_fix_map.keys())
        print(missing_from_map if missing_from_map else "  None")
    
        print("\nIDs in your map but NOT found in dataset:")
        extra_in_map = set(logbook_id_fix_map.keys()) - set(trunc_ids)
        print(extra_in_map if extra_in_map else "  None")
    
    else:
        print("Mapping length matches the number of truncated IDs.")
    
    processed_tiers[tier] = current_df

In [ ]:
processed_tiers = {}
from utils.meta import logbook_id_fix_map, standardize_logbook_id

for i in range(4):
    tier = i + 1
    current_df = export_finalized[tier].copy()
    
    # ensure datetime conversion for filtering
    current_df['Entry Date'] = pd.to_datetime(current_df['Entry Date'])
    
    
    # apply corrections to truncated ID map
    current_df["LogBook ID"] = current_df["LogBook ID"].replace(logbook_id_fix_map)
    
    # handle Gage H. Phillips voyage splitting
    ghp_trunc = "Gage H. Phillips (Schooner) 18…"
    ghp_mask = current_df["LogBook ID"] == ghp_trunc
    ghp_years = current_df.loc[ghp_mask, "Entry Date"].dt.year

    # apply voyage specific masks
    current_df.loc[ghp_mask & ghp_years.between(1878, 1880), "LogBook ID"] = \
        "Gage H. Phillips (Schooner) 1878-1880"
    current_df.loc[ghp_mask & ghp_years.between(1881, 1883), "LogBook ID"] = \
        "Gage H. Phillips (Schooner) 1881-1883"
    
    # standardize strings
    current_df["LogBook ID"] = current_df["LogBook ID"].astype(str).apply(standardize_logbook_id)
    
    #  apply specific date fix map
    date_fix_map = {
        "Arab (Ship) 1849-1853" : "Arab (Ship) 1849-1852",
        "Brunswick (Ship)"      : "Brunswick (Ship) 1861-1862",
        "Catalpa (Bark)"        : "Catalpa (Bark) 1875-1876",
        "Perry (Bark)"          : "Perry (Bark) 1877-1880"
    }
    current_df["LogBook ID"] = current_df["LogBook ID"].replace(date_fix_map)
    
    #  store result
    processed_tiers[tier] = current_df
    print(f"Tier {tier} cleaning complete.")

Trim out extra columns - this is the dataset we are publishing

In [ ]:
# just keeping relevant wind vars
#and reading this out as our final dataset
keep_cols = [
    'ID', 'LogBook ID', 'Entry Date', 'Latitude',
    'Latitude_decimal', 'Longitude', 'Longitude_decimal', 
    'Wind Direction', 'Clean Wind Direction', 'WD_Bearing', 'Wind Speed/Force', 'BF Value']

wind_dir = os.path.join(base_dir, "processed_wind_data")
os.makedirs(wind_dir, exist_ok=True)

wind_df = {}
for i in range(4):
    tier = i+1
    wind_df[tier] = processed_tiers[tier][keep_cols].copy()

    wind_df[tier].to_csv(f"processed_wind_data/Tier{tier}_winds.csv", index=False)

In [ ]:
df_final = processed_tiers[4].copy()

grouped = (
    df_final
    .groupby('LogBook ID')
    .agg(
        Total_Entries = ('LogBook ID', "size"),
        Total_Usable_Entries = ('Tier4_usable', "sum"), 
    )
    .reset_index()
)

summary_df = grouped.copy()
summary_df

### Loading in metadata

In [ ]:
#read in meta data file
meta_df = pd.read_csv(os.path.join(base_dir, f'csv_files/logbooks-export-meta.csv'), low_memory=False)

df = meta_df.copy()
df["LogBook ID"] = df["LogBook ID"].astype(str).apply(standardize_logbook_id)

In [ ]:
ID_COL = "LogBook ID"     
RESEARCHER_COL = "Researcher"   
REPOSITORY_COL = "Repository" 

meta_subset = df[[ID_COL, RESEARCHER_COL, REPOSITORY_COL]].copy()

# All duplicated IDs (including the first occurrence)
dupes_all = df[df.duplicated(subset=[ID_COL], keep=False)]

dupes_all.sort_values(by=ID_COL)

# one row per logbook in metadata
meta_subset = meta_subset.drop_duplicates(subset=["LogBook ID"])

In [ ]:
standard_repo_map = {'NBWM' : 'New Bedford Whaling Museum',
                     'PPL' : 'Providence Public Library',
                     'Falmouth Museums on the Green' : 'Falmouth Museum on the Green',
                     'Providence Public library' : 'Providence Public Library',
                     'Nicholson Whaling Collection' : 'Providence Public Library',
                     'Nicholson Whaling Collection, Providence Public Library' : 
                                      'Providence Public Library'}

meta_subset[REPOSITORY_COL] = meta_subset[REPOSITORY_COL].replace(standard_repo_map)

In [ ]:
summary_with_meta = summary_df.merge(
    meta_subset,
    on="LogBook ID",
    how="left"   # keep all logbooks in summary_df, even if metadata missing
)

#Uncomment to create - just dont need to overwrite every time
summary_with_meta.to_csv(f"csv_files/Tier{tier}logbooks_meta.csv", index=False)
summary_with_meta

In [ ]:
# Logbooks in summary_df with no matching metadata
missing_meta = summary_with_meta[summary_with_meta["Researcher"].isna() &
                                 summary_with_meta["Repository"].isna()]

if not missing_meta.empty:
    print("No metadata found for these LogBook IDs:")
    print(missing_meta["LogBook ID"].to_list())
else:
    print("All LogBook IDs in summary_df have matching metadata.")

# Logbooks in meta_df that never appear in summary_df (optional)
summary_ids = set(summary_df["LogBook ID"])
meta_ids = set(meta_subset["LogBook ID"])

missing_in_dataset = meta_ids - summary_ids
if missing_in_dataset:
    print("These logbooks are currently excluded from the data set:")
    print(list(missing_in_dataset))

#### Step 2:
- Generate library of visualizations for each voyage as well as the combined dataset (all voyages combined). These will be attached to the summary stats table.

In [ ]:
from utils.meta import plot_logbook_with_options

#select dataframe to plot from
tier = 4
plot_df = df_final.copy()

#select logbook to plot
selected_logbook = 'Yeoman (Bark) 1843-1848'

cwd = os.getcwd()

In [ ]:
#single logbook inspection

plot_logbook_with_options(plot_df, selected_logbook, save=False, color_by = 'BF Value',  title=f'Voyage – {selected_logbook}', plot_path=True)

In [ ]:
#plot_logbook_with_options(plot_df, selected_logbook, save=False, color_by = 'green', plot_path = False, title=f'Voyage – {selected_logbook}')

In [ ]:
#create all figs
logbook_list = plot_df['LogBook ID'].unique().tolist()
logbooks_to_rm = ['Ann (Ship) 1841', 'Ann (Ship) 1791-1792', 'Sea Breeze (1865 - 1871)'] #these contain no valid entries

for logbook in logbooks_to_rm:
    if logbook in logbook_list:
        logbook_list.remove(logbook)

save_dir = os.path.join(cwd, "meta_figs/single_voyages/voyage_progress")
os.makedirs(save_dir, exist_ok=True)

for logbook in logbook_list:
    plot_logbook_with_options(plot_df, logbook, save=True, save_path = save_dir, selected_tier = tier, color_by = 'Entry Date', plot_path=True, title=f'{logbook}')

In [ ]:
#plot_logbook_with_options(plot_df, logbook, save=True, save_path = save_dir, selected_tier = tier, color_by = 'BF Value', plot_path=True, title=f'Wind Force Values – {logbook}')

In [ ]:
save_dir = os.path.join(cwd, "meta_figs/single_voyages/wind_force")
os.makedirs(save_dir, exist_ok=True)

for logbook in logbook_list:
    plot_logbook_with_options(plot_df, logbook, save=True, save_path = save_dir, selected_tier = tier, color_by = 'BF Value', plot_path=True, title=f'Wind Force Values – {logbook}')

In [ ]:
from utils.meta import plot_all_logbooks_with_options

In [ ]:
save_dir = os.path.join(cwd, "meta_figs/combined_voyages")
os.makedirs(save_dir, exist_ok=True)

plot_all_logbooks_with_options(plot_df, save=True, save_path = save_dir, selected_tier = tier, color_by = 'BF Value', title=f'Wind Force Values – All Usable Entries')

In [ ]:
plot_all_logbooks_with_options(plot_df, save=True, save_path = save_dir, selected_tier = tier, color_by = 'WD_Bearing', title=f'Wind Bearing – All Usable Entries')

In [ ]:
#sync all figs and csv to Drive
!utils/sync_metadata.sh 